In [1]:
#cell 1:

!wget https://sumith1896.github.io/spoc/data/spoc.zip


--2025-11-19 18:16:43--  https://sumith1896.github.io/spoc/data/spoc.zip
Resolving sumith1896.github.io (sumith1896.github.io)... 185.199.109.153, 185.199.110.153, 185.199.108.153, ...
Connecting to sumith1896.github.io (sumith1896.github.io)|185.199.109.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9960355 (9.5M) [application/x-zip-compressed]
Saving to: ‘spoc.zip’

spoc.zip            100%[===================>]   9.50M  --.-KB/s    in 0.09s   

2025-11-19 18:16:43 (105 MB/s) - ‘spoc.zip’ saved [9960355/9960355]



In [ ]:
#cell 2:
!rm /kaggle/working/spoc_program_level.csv

In [ ]:
#cell 3:
!rm -rf /kaggle/working/dataset

In [2]:
#cell 4:
!unzip spoc.zip -d dataset

Archive:  spoc.zip
   creating: dataset/test/
  inflating: dataset/test/spoc-testw.tsv  
  inflating: dataset/test/spoc-testp.tsv  
   creating: dataset/testcases/
   creating: dataset/testcases/1000A/
  inflating: dataset/testcases/1000A/1000A_testcases.txt  
  inflating: dataset/testcases/1000A/1000A_testcases_hidden.txt  
  inflating: dataset/testcases/1000A/1000A_testcases_public.txt  
   creating: dataset/testcases/1003A/
  inflating: dataset/testcases/1003A/1003A_testcases.txt  
  inflating: dataset/testcases/1003A/1003A_testcases_hidden.txt  
  inflating: dataset/testcases/1003A/1003A_testcases_public.txt  
   creating: dataset/testcases/1004A/
  inflating: dataset/testcases/1004A/1004A_testcases.txt  
  inflating: dataset/testcases/1004A/1004A_testcases_hidden.txt  
  inflating: dataset/testcases/1004A/1004A_testcases_public.txt  
   creating: dataset/testcases/1005A/
  inflating: dataset/testcases/1005A/1005A_testcases.txt  
  inflating: dataset/testcases/1005A/1005A_testcases

In [3]:
#cell 5:
!ls dataset

LICENSE  README.md  test  testcases  train


In [4]:
#cell 6:
!ls dataset/train | head

split
spoc-train.tsv


In [5]:
#cell 7:
!ls dataset/train/split

spoc-train-eval.tsv  spoc-train-test.tsv  spoc-train-train.tsv


In [2]:
#cell 8:
import pandas as pd

train_df = pd.read_csv("dataset/train/split/spoc-train-train.tsv", sep="\t")
train_df.head(5)

,text,code,workerid,probid,subid,line,indent
0,NaN,int main() {,1,3A,41470897,0,0
1,create string s,string s;,1,3A,41470897,1,1
2,"create integers x1, y1, x2, y2","int x1, y1, x2, y2;",1,3A,41470897,2,1
3,read s,cin >> s;,1,3A,41470897,3,1
4,set x1 to s[0] - 96,x1 = s[0] - 96;,1,3A,41470897,4,1


In [3]:
#cell 9:
train_df.columns

Index(['text', 'code', 'workerid', 'probid', 'subid', 'line', 'indent'], dtype='object')

In [4]:
#cell 10:
import pandas as pd

# 1️⃣ Load the dataset with EXPLICIT column names
df = pd.read_csv(
    "dataset/train/split/spoc-train-train.tsv",
    sep="\t",
    header=0,
    names=["text", "code", "workerid", "probid", "subid", "line", "indent"],  # ✅ Add this line
    skiprows=[0],
    dtype=str,
    keep_default_na=False
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)

Dataset shape: (246085, 7)
Columns: ['text', 'code', 'workerid', 'probid', 'subid', 'line', 'indent']


,text,code,workerid,probid,subid,line,indent
0,create string s,string s;,01,3A,41470897,1,1
1,"create integers x1, y1, x2, y2","int x1, y1, x2, y2;",01,3A,41470897,2,1
2,read s,cin >> s;,01,3A,41470897,3,1
3,set x1 to s[0] - 96,x1 = s[0] - 96;,01,3A,41470897,4,1
4,set y1 to s[1] - '0',y1 = s[1] - '0';,01,3A,41470897,5,1
5,read s,cin >> s;,01,3A,41470897,6,1
6,set x2 to s[0] - 96,x2 = s[0] - 96;,01,3A,41470897,7,1
7,set y2 to s[1] - '0',y2 = s[1] - '0';,01,3A,41470897,8,1
8,print maximum of absolute value of x1 - x2 and...,"cout << max(abs(x1 - x2), abs(y1 - y2)) << endl;",01,3A,41470897,9,1
9,while x1 is not x2 or y1 is not y2,while (x1 != x2 || y1 != y2) {,01,3A,41470897,10,1


In [5]:
#cell 11:
grouped_df = (
    df.groupby(["probid", "subid"], sort=False)
      .apply(lambda g: pd.Series({
          "pseudocode": "\n".join([x for x in g["text"] if x.strip() != ""]),
          "code": "\n".join(g["code"])
      }))
      .reset_index()
)

print("✅ Program-level pairs created:", len(grouped_df))
grouped_df.head(5)

✅ Program-level pairs created: 11528


/tmp/ipykernel_48/1770592424.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,probid,subid,pseudocode,code
0,3A,41470897,"create string s\ncreate integers x1, y1, x2, y...","string s;\nint x1, y1, x2, y2;\ncin >> s;\nx1 ..."
1,3A,48266012,create character array c1 of size 2\ncreate ch...,int main() {\nchar c1[2];\nchar c2[2];\ncin >>...
2,3A,48293169,"n, m = long long integers\nx, y, a, b = charac...","long long n, m;\nchar x, y, a, b;\nstring s, s..."
3,3A,47102964,"create strings now, goal, create string array ...","int main() {\nstring now, goal, way[100];\nint..."
4,3A,46040797,create string S\nread S\ncreate integers x1 to...,int main() {\nstring S;\ncin >> S;\nint x1 = S...


In [6]:
#cell 12:
sample = grouped_df.sample(1).iloc[0]
print("🧩 PSEUDOCODE:\n")
print(sample.pseudocode)
print("\n💻 CODE:\n")
print(sample.code)


🧩 PSEUDOCODE:

let n be a integer
read n
b = 2d array of integers with n rows and 2 columns respectively
for i = 0 to n exclusive
for j = 0 to 2 exclusive , read b[i][j]
let count be a integer with count = 0
for i = 0 to n exclusive
for j = 0 to n exclusive
if b[i][0] is equal to b[j][1] and i is not equal to j , increment count by 1
print count and newline

💻 CODE:

int main() {
int n;
cin >> n;
int b[n][2];
for (int i = 0; i < n; i++) {
for (int j = 0; j < 2; j++) { cin >> b[i][j]; }
}
int count = 0;
for (int i = 0; i < n; i++) {
for (int j = 0; j < n; j++) {
if (b[i][0] == b[j][1] && i != j) { count += 1; }
}
}
cout << count << endl;
}


In [11]:
#cell 13:
# Save the cleaned, grouped dataset
grouped_df.to_csv("spoc_program_level.csv", index=False, encoding="utf-8")

print("💾 Saved program-level dataset successfully!")
print("✅ Total programs saved:", len(grouped_df))


💾 Saved program-level dataset successfully!
✅ Total programs saved: 11528


In [7]:
#cell 14:
# Check saved data
test_df = pd.read_csv("spoc_program_level.csv")
print("Loaded:", len(test_df), "rows")

test_df.head(2)



Loaded: 11528 rows


,probid,subid,pseudocode,code
0,3A,41470897,"create string s\ncreate integers x1, y1, x2, y...","string s;\nint x1, y1, x2, y2;\ncin >> s;\nx1 ..."
1,3A,48266012,create character array c1 of size 2\ncreate ch...,int main() {\nchar c1[2];\nchar c2[2];\ncin >>...


In [8]:
# cell 15:
# Combine pseudocode and code into a single formatted text string
grouped_df["text"] = (
    "### Pseudocode:\n"
    + grouped_df["pseudocode"].astype(str)
    + "\n\n### Code:\n"
    + grouped_df["code"].astype(str)
)

print("Processing complete: pseudocode and code combined into grouped_df['text'].")


Processing complete: pseudocode and code combined into grouped_df['text'].


In [9]:
# Apply deduplication to your already-grouped data
def remove_duplicate_lines(text):
    lines = text.split('\n')
    seen = set()
    unique_lines = []
    for line in lines:
        if line.strip() not in seen:
            unique_lines.append(line)
            seen.add(line.strip())
    return '\n'.join(unique_lines)

grouped_df['pseudocode'] = grouped_df['pseudocode'].apply(remove_duplicate_lines)
grouped_df['code'] = grouped_df['code'].apply(remove_duplicate_lines)

print("Deduplication complete: pseudocode and code cleaned.")


Deduplication complete: pseudocode and code cleaned.


In [10]:
#cell 17: showing example
sample = grouped_df[
    (grouped_df["probid"] == "3A") & (grouped_df["subid"] == "41470897")
].iloc[0]

print("🧩 PSEUDOCODE:\n", sample["pseudocode"])
print("\n💻 CODE:\n", sample["code"])




🧩 PSEUDOCODE:
 create string s
create integers x1, y1, x2, y2
read s
set x1 to s[0] - 96
set y1 to s[1] - '0'
set x2 to s[0] - 96
set y2 to s[1] - '0'
print maximum of absolute value of x1 - x2 and absolute value of y1 - y2, print newline
while x1 is not x2 or y1 is not y2
if x2 is greater than x1
print "R"
increment x1
if x2 is less than x1
print "L"
decrement x1
if y1 is greater than y2
print "D"
decrement y1
if y1 is less than y2
print "U"
increment y1
print "\n"
s = string
x1, y1, x2, y2 = integers
Read s
set x1 = s[0] - 96
set y1 = s[1] - 0
set x2 = s[0] - 96
set y2 = s[1] - 0
print maximum of absolute of (x1 - x2) and absolute of (y1 - y2)
while (x1 != x2 OR y1 != y2) is TRUE, do the following
if x2 is greater than x1 , do the following
print R
if x2 is less than x1 , do the following
print L
if y1 is greater than y2 , do the following
print D
if y1 is less than y2 , do the following
print U
print a new line

💻 CODE:
 string s;
int x1, y1, x2, y2;
cin >> s;
x1 = s[0] - 96;
y1 = s

In [11]:
#cell 18:
# 💾 Save the cleaned and formatted dataset 

# First, check if we actually have a dummy row
print("Before any drops - Dataset shape:", grouped_df.shape)
print("First row preview:")
print("Pseudocode:", str(grouped_df.iloc[0]["pseudocode"])[:100] + "...")
print("Code:", str(grouped_df.iloc[0]["code"])[:100] + "...")

# Only drop if it's actually a dummy/empty row
if (grouped_df.iloc[0]["pseudocode"].strip() == "" or 
    len(grouped_df.iloc[0]["pseudocode"]) < 5):  # If very short/empty
    grouped_df = grouped_df.iloc[1:].reset_index(drop=True)
    print("✅ Dropped dummy first row")
else:
    print("✅ First row looks valid - keeping it")

# Create formatted text
grouped_df["text"] = (
    "### Pseudocode:\n" +
    grouped_df["pseudocode"].astype(str) +
    "\n\n### Code:\n" +
    grouped_df["code"].astype(str)
)

# Save only 'text' column
final_df = grouped_df[["text"]].copy()
final_df.to_csv("spoc_gpt2_clean_formatted.csv", index=False)

print("✅ Saved clean GPT-2 formatted dataset as 'spoc_gpt2_clean_formatted.csv'")
print("Total programs saved:", len(final_df))

Before any drops - Dataset shape: (11528, 5)
First row preview:
Pseudocode: create string s
create integers x1, y1, x2, y2
read s
set x1 to s[0] - 96
set y1 to s[1] - '0'
set x...
Code: string s;
int x1, y1, x2, y2;
cin >> s;
x1 = s[0] - 96;
y1 = s[1] - '0';
x2 = s[0] - 96;
y2 = s[1] -...
✅ First row looks valid - keeping it
✅ Saved clean GPT-2 formatted dataset as 'spoc_gpt2_clean_formatted.csv'
Total programs saved: 11528


In [12]:
#cell 20:
# 📦 FORCE INSTALL correct versions (override Kaggle defaults)

print("🔄 Force installing correct package versions...")

# Use --force-reinstall and --no-cache-dir to truly override Kaggle's versions
!pip install --force-reinstall --no-cache-dir --no-deps \
    huggingface-hub==0.20.1

!pip install --force-reinstall --no-cache-dir \
    tokenizers==0.15.0 \
    safetensors==0.4.2

!pip install --no-cache-dir \
    transformers==4.36.2 \
    accelerate==0.26.1 \
    peft==0.8.2 \
    datasets

print("\n✅ Packages installed!")
print("⚠️ NOW YOU MUST RESTART THE KERNEL!")
print("After restart, run from Cell 21 (skip Cell 20)")

🔄 Force installing correct package versions...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 6.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 35.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 270.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 356.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 331.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 294.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 266.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 353.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [13]:
#cell 21:
import pandas as pd
# Load dataset with pandas (no PyArrow dependency)
print("📊 Loading dataset with pandas...")
df = pd.read_csv("spoc_gpt2_clean_formatted.csv")

# Display dataset info
print(f"Dataset shape: {df.shape}")
print(f"Number of examples: {len(df)}")
print("Sample example:")
print(df['text'].iloc[0][:2000] + "...")  # Show first 200 chars

📊 Loading dataset with pandas...
Dataset shape: (11528, 1)
Number of examples: 11528
Sample example:
### Pseudocode:
create string s
create integers x1, y1, x2, y2
read s
set x1 to s[0] - 96
set y1 to s[1] - '0'
set x2 to s[0] - 96
set y2 to s[1] - '0'
print maximum of absolute value of x1 - x2 and absolute value of y1 - y2, print newline
while x1 is not x2 or y1 is not y2
if x2 is greater than x1
print "R"
increment x1
if x2 is less than x1
print "L"
decrement x1
if y1 is greater than y2
print "D"
decrement y1
if y1 is less than y2
print "U"
increment y1
print "\n"
s = string
x1, y1, x2, y2 = integers
Read s
set x1 = s[0] - 96
set y1 = s[1] - 0
set x2 = s[0] - 96
set y2 = s[1] - 0
print maximum of absolute of (x1 - x2) and absolute of (y1 - y2)
while (x1 != x2 OR y1 != y2) is TRUE, do the following
if x2 is greater than x1 , do the following
print R
if x2 is less than x1 , do the following
print L
if y1 is greater than y2 , do the following
print D
if y1 is less than y2 , do the follo

In [14]:
#cell 22:
# Import required modules
import torch
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Add padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token

# Tokenization function for pandas data
def tokenize_texts(texts):
    """
    Tokenize the text data for causal language modeling using pandas
    """
    # Tokenize the entire text sequence
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length", 
        max_length=1024,
        return_tensors="pt"
    )
    
    # For causal LM, labels are the same as input_ids
    tokenized["labels"] = tokenized["input_ids"].clone()
    
    return tokenized

# Apply tokenization to our pandas data
print("🔄 Tokenizing dataset...")

# Convert texts to list from pandas DataFrame
texts = df['text'].tolist()

# Tokenize in batches to avoid memory issues
batch_size = 1000
all_tokenized = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    tokenized_batch = tokenize_texts(batch_texts)
    all_tokenized.append(tokenized_batch)
    print(f"✅ Tokenized batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")

print("✅ Tokenization complete!")

# Combine all batches
input_ids = torch.cat([batch['input_ids'] for batch in all_tokenized])
attention_mask = torch.cat([batch['attention_mask'] for batch in all_tokenized])
labels = torch.cat([batch['labels'] for batch in all_tokenized])

print(f"📊 Tokenized data shapes:")
print(f"Input IDs: {input_ids.shape}")
print(f"Attention Mask: {attention_mask.shape}")
print(f"Labels: {labels.shape}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

🔄 Tokenizing dataset...
✅ Tokenized batch 1/12
✅ Tokenized batch 2/12
✅ Tokenized batch 3/12
✅ Tokenized batch 4/12
✅ Tokenized batch 5/12
✅ Tokenized batch 6/12
✅ Tokenized batch 7/12
✅ Tokenized batch 8/12
✅ Tokenized batch 9/12
✅ Tokenized batch 10/12
✅ Tokenized batch 11/12
✅ Tokenized batch 12/12
✅ Tokenization complete!
📊 Tokenized data shapes:
Input IDs: torch.Size([11528, 1024])
Attention Mask: torch.Size([11528, 1024])
Labels: torch.Size([11528, 1024])


In [31]:
#cell 23:
# Split dataset into training and validation
print("🔄 Splitting dataset...")

# Calculate split sizes
train_size = int(0.9 * len(input_ids))
eval_size = len(input_ids) - train_size

# Split the tensors
train_input_ids = input_ids[:train_size]
train_attention_mask = attention_mask[:train_size]
train_labels = labels[:train_size]

eval_input_ids = input_ids[train_size:]
eval_attention_mask = attention_mask[train_size:]
eval_labels = labels[train_size:]

print(f"✅ Dataset split complete!")
print(f"📊 Training samples: {len(train_input_ids):,}")
print(f"📊 Validation samples: {len(eval_input_ids):,}")
print(f"📈 Train/Eval ratio: {len(train_input_ids)/len(eval_input_ids):.2f}:1")

# Create a simple dictionary to mimic dataset structure
train_dataset = {
    'input_ids': train_input_ids,
    'attention_mask': train_attention_mask,
    'labels': train_labels
}

eval_dataset = {
    'input_ids': eval_input_ids,
    'attention_mask': eval_attention_mask,
    'labels': eval_labels
}

print("✅ Datasets prepared successfully!")

🔄 Splitting dataset...
✅ Dataset split complete!
📊 Training samples: 10,375
📊 Validation samples: 1,153
📈 Train/Eval ratio: 9.00:1
✅ Datasets prepared successfully!


In [32]:
#cell 24a:
# Verify all imports work correctly
print("🔍 Verifying package installations...")

try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
    
    import transformers
    print(f"✅ Transformers: {transformers.__version__}")
    
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("✅ Transformers imports: OK")
    
    import peft
    print(f"✅ PEFT: {peft.__version__}")
    
    import accelerate
    print(f"✅ Accelerate: {accelerate.__version__}")
    
    print("\n✅ All packages verified and ready!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n⚠️ If you see errors, make sure you RESTARTED THE KERNEL after Cell 20!")

🔍 Verifying package installations...
✅ PyTorch: 2.6.0+cu124
✅ Transformers: 4.53.3
✅ Transformers imports: OK
✅ PEFT: 0.16.0
✅ Accelerate: 1.9.0

✅ All packages verified and ready!


In [33]:
#cell 25:
# 🧩 Step 5 — Load Model & Apply LoRA

# Load base GPT-2 model
print("🔄 Loading GPT-2 model...")
try:
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    print("✅ Base GPT-2 model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Trying alternative loading method...")
    # Alternative: Load with specific config
    from transformers import GPT2Config, GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained("gpt2")

print(f"📊 Total parameters: {model.num_parameters():,}")

# Configure LoRA for code geneceration
print("🔄 Applying LoRA configuration...")
try:
    from peft import LoraConfig, get_peft_model
    
    lora_config = LoraConfig(
        r=8,                # Rank of adaptation matrices
        lora_alpha=32,      # Scaling factor
        lora_dropout=0.1,   # Dropout for regularization
        bias="none",        # Don't train bias parameters
        target_modules=["c_attn", "c_proj"],  # GPT-2 attention layers
        task_type="CAUSAL_LM",  # Causal language modeling
    )

    # Apply LoRA to the model
    model = get_peft_model(model, lora_config)
    print("✅ LoRA applied successfully!")
    
except ImportError:
    print("❌ PEFT not available, using regular fine-tuning")
    # Freeze layers instead of using LoRA
    for param in model.transformer.h[:8].parameters():
        param.requires_grad = False
    print("✅ Applied layer freezing instead of LoRA")

print("\n📊 Parameter Summary:")
try:
    model.print_trainable_parameters()
except:
    total_params = model.num_parameters()
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Percentage: {trainable_params/total_params*100:.2f}%")

print("\n🔍 Model verification:")
print(f"Model vocab size: {model.config.vocab_size}")
print("✅ Model setup complete - ready for training!")

🔄 Loading GPT-2 model...


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Base GPT-2 model loaded successfully!
📊 Total parameters: 124,439,808
🔄 Applying LoRA configuration...
✅ LoRA applied successfully!

📊 Parameter Summary:
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475

🔍 Model verification:
Model vocab size: 50257
✅ Model setup complete - ready for training!


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


In [34]:
#cell 26:
# Install minimal metrics packages that don't depend on datasets
!pip install sacrebleu

import sacrebleu
import numpy as np

print("✅ Minimal metrics packages installed!")

# Simple evaluation metrics without evaluate library
def compute_metrics(eval_pred):
    """
    Simple evaluation metrics without PyArrow dependencies
    Uses basic accuracy and sacrebleu only
    """
    logits, labels = eval_pred
    
    metrics = {}
    
    # Calculate basic accuracy (memory efficient)
    predictions = logits.argmax(-1)
    
    # Mask out padding tokens (ignore_index=-100)
    mask = labels != -100
    correct = (predictions == labels) & mask
    accuracy = correct.sum() / mask.sum()
    metrics["accuracy"] = round(accuracy.item() * 100, 2)
    
    # Calculate loss
    try:
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            labels.view(-1), 
            ignore_index=-100
        )
        metrics["eval_loss"] = round(loss.item(), 4)
        metrics["perplexity"] = round(torch.exp(loss).item(), 2)
    except Exception as e:
        print(f"Loss computation error: {e}")
        metrics["eval_loss"] = 0.0
        metrics["perplexity"] = 0.0
    
    # Simple BLEU calculation on a small sample (to save memory)
    try:
        # Sample a small subset for BLEU calculation
        sample_size = min(50, len(logits))
        indices = np.random.choice(len(logits), sample_size, replace=False)
        
        sample_logits = logits[indices]
        sample_labels = labels[indices]
        
        sample_predictions = sample_logits.argmax(-1)
        
        # Decode only the sampled data
        decoded_preds = tokenizer.batch_decode(sample_predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(sample_labels, skip_special_tokens=True)
        
        # Calculate BLEU using sacrebleu
        bleu_score = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
        metrics["sacrebleu"] = round(bleu_score.score, 2)
        
    except Exception as e:
        print(f"BLEU computation error: {e}")
        metrics["sacrebleu"] = 0.0
    
    print(f"📊 Evaluation - Accuracy: {metrics['accuracy']}%, BLEU: {metrics.get('sacrebleu', 0)}, Loss: {metrics['eval_loss']:.4f}")
    
    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return metrics



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✅ Minimal metrics packages installed!


In [25]:
# First, let's fix the installation completely
!pip install --upgrade transformers==4.36.0 torch>=2.0.0 --quiet

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


^C
ERROR: Operation cancelled by user


In [38]:
# 🧩 Step 7 — Configure Training (OPTIMIZED FOR SPEED)

import torch
from transformers import DataCollatorForLanguageModeling

# Import TrainingArguments
try:
    from transformers import TrainingArguments
except ImportError as e:
    print(f"Import error: {e}")
    print("Applying manual fix...")
    
    import transformers.pytorch_utils
    if not hasattr(transformers.pytorch_utils, 'is_torch_greater_or_equal_than_2_8'):
        def is_torch_greater_or_equal_than_2_8():
            return True
        transformers.pytorch_utils.is_torch_greater_or_equal_than_2_8 = is_torch_greater_or_equal_than_2_8
    
    from transformers import TrainingArguments

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# OPTIMIZED Training arguments for 2-2.5 hours
training_args = TrainingArguments(
    output_dir="./spoc_gpt2_lora_finetuned",
    overwrite_output_dir=True,
    num_train_epochs=2,                    # Reduced from 3 to 2
    per_device_train_batch_size=4,         # Doubled from 2 to 4
    per_device_eval_batch_size=2,          # Increased from 1 to 2
    gradient_accumulation_steps=1,
    eval_strategy="epoch", 
    save_strategy="epoch",
    logging_steps=100,                     # Less frequent logging
    learning_rate=8e-5,                    # Slightly higher LR for faster convergence
    weight_decay=0.01,
    warmup_steps=50,                       # Reduced warmup
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=True,            # Enabled for speed
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    remove_unused_columns=True,
    dataloader_num_workers=2,              # Parallel data loading
    group_by_length=True,                  # Faster training
)

print("✅ OPTIMIZED Training arguments configured for 2-2.5 hours!")

✅ OPTIMIZED Training arguments configured for 2-2.5 hours!


In [39]:
#cell 28:
# 🧩 Step 8 — Progress Monitoring Callback

# Import required components
from transformers import TrainerCallback
import time

class TrainingProgressCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None
        self.total_steps = None
    
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.total_steps = state.max_steps
        print("🚀 TRAINING STARTED!")
        print("=" * 60)
        print(f"📊 Total examples: {len(train_dataset):,}")
        print(f"📈 Total steps: {state.max_steps:,}")
        print(f"📚 Total epochs: {args.num_train_epochs}")
        print(f"📦 Batch size: {args.per_device_train_batch_size}")
        print(f"🎯 Trainable parameters: {model.num_parameters(only_trainable=True):,}")
        print("=" * 60)
    
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0:  # Every 50 steps
            progress = (state.global_step / self.total_steps) * 100
            elapsed = time.time() - self.start_time
            
            print(f"\n📈 Step {state.global_step:,}/{self.total_steps:,} ({progress:.1f}%)")
            
            # Time estimation
            if state.global_step > 10:
                time_per_step = elapsed / state.global_step
                remaining = (self.total_steps - state.global_step) * time_per_step
                hours = int(remaining // 3600)
                minutes = int((remaining % 3600) // 60)
                print(f"⏱️  ETA: {hours:02d}h {minutes:02d}m remaining")
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs:
            if 'loss' in logs and state.global_step % 50 == 0:
                print(f"💔 Loss: {logs['loss']:.4f}")
            if 'learning_rate' in logs and state.global_step % 100 == 0:
                print(f"📚 LR: {logs['learning_rate']:.2e}")
    
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            print(f"\n🎉 EPOCH {int(state.epoch)} COMPLETED!")
            print("=" * 50)

progress_callback = TrainingProgressCallback()

print("✅ Progress callback configured!")

✅ Progress callback configured!


In [40]:
#cell 29:
# 🧩 Step 9 — Custom Training Loop (BALANCED OPTIMIZATION)

print("🎯 Starting BALANCED custom training loop...")

import time
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# Convert dictionary datasets to TensorDataset
print("🔄 Converting datasets to proper format...")

train_tensor_dataset = TensorDataset(
    train_dataset['input_ids'],
    train_dataset['attention_mask'], 
    train_dataset['labels']
)

eval_tensor_dataset = TensorDataset(
    eval_dataset['input_ids'],
    eval_dataset['attention_mask'],
    eval_dataset['labels']
)

print(f"✅ Created TensorDatasets:")
print(f"  - Train: {len(train_tensor_dataset)} samples")
print(f"  - Eval: {len(eval_tensor_dataset)} samples")

# OPTIMIZED: Increase batch size but keep 3 epochs
train_loader = DataLoader(train_tensor_dataset, batch_size=4, shuffle=True)  # 2 → 4
eval_loader = DataLoader(eval_tensor_dataset, batch_size=2, shuffle=False)   # 1 → 2

print(f"📊 DataLoaders created:")
print(f"  - Train batches: {len(train_loader)}")
print(f"  - Eval batches: {len(eval_loader)}")

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"🔧 Using device: {device}")

# OPTIMIZED: Slightly higher learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5)  # 5e-5 → 6e-5

# Training loop - KEEP 3 EPOCHS for better results
model.train()
total_steps = len(train_loader) * 3  # KEEP 3 epochs
global_step = 0
start_time = time.time()

print("\n🚀 BALANCED TRAINING STARTED!")
print("=" * 60)
print(f"📊 Total examples: {len(train_tensor_dataset):,}")
print(f"📈 Total steps: {total_steps:,}")
print(f"📚 Total epochs: 3")  # KEEP 3 for better quality
print(f"📦 Batch size: 4")    # Increased for speed
print(f"🎯 Learning rate: 6e-5")  # Optimized
print(f"🎯 Trainable parameters: {model.num_parameters(only_trainable=True):,}")
print("=" * 60)

# KEEP 3 EPOCHS for better model quality
for epoch in range(3):  # KEEP 3 epochs
    print(f"\n📚 Starting Epoch {epoch + 1}/3")
    
    epoch_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        # Unpack the batch (input_ids, attention_mask, labels)
        input_ids, attention_mask, labels = batch
        
        # Move batch to device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        global_step += 1
        
        # OPTIMIZED: Less frequent logging
        if global_step % 100 == 0 or (batch_idx == 0 and global_step == 1):
            progress = (global_step / total_steps) * 100
            elapsed = time.time() - start_time
            
            print(f"\n📈 Step {global_step:,}/{total_steps:,} ({progress:.1f}%)")
            print(f"💔 Loss: {loss.item():.4f}")
            
            # Time estimation
            if global_step > 10:
                time_per_step = elapsed / global_step
                remaining = (total_steps - global_step) * time_per_step
                hours = int(remaining // 3600)
                minutes = int((remaining % 3600) // 60)
                print(f"⏱️  ETA: {hours:02d}h {minutes:02d}m remaining")
    
    # End of epoch
    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"\n🎉 EPOCH {epoch + 1} COMPLETED!")
    print(f"📊 Average Loss: {avg_epoch_loss:.4f}")
    print("=" * 50)
    
    # OPTIMIZED: Faster evaluation
    model.eval()
    eval_loss = 0
    eval_samples = min(50, len(eval_loader))  # Reduced from 100
    with torch.no_grad():
        for i, batch in enumerate(eval_loader):
            if i >= eval_samples:
                break
                
            # Unpack the batch
            input_ids, attention_mask, labels = batch
            
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            eval_loss += outputs.loss.item()
    
    avg_eval_loss = eval_loss / eval_samples
    print(f"🧪 Validation Loss: {avg_eval_loss:.4f}")
    model.train()

print("✅ BALANCED Training completed successfully!")

🎯 Starting BALANCED custom training loop...
🔄 Converting datasets to proper format...
✅ Created TensorDatasets:
  - Train: 10375 samples
  - Eval: 1153 samples
📊 DataLoaders created:
  - Train batches: 2594
  - Eval batches: 577
🔧 Using device: cuda

🚀 BALANCED TRAINING STARTED!
📊 Total examples: 10,375
📈 Total steps: 7,782
📚 Total epochs: 3
📦 Batch size: 4
🎯 Learning rate: 6e-5
🎯 Trainable parameters: 811,008

📚 Starting Epoch 1/3

📈 Step 1/7,782 (0.0%)
💔 Loss: 0.4513

📈 Step 100/7,782 (1.3%)
💔 Loss: 0.3921
⏱️  ETA: 02h 40m remaining

📈 Step 200/7,782 (2.6%)
💔 Loss: 0.3341
⏱️  ETA: 02h 38m remaining

📈 Step 300/7,782 (3.9%)
💔 Loss: 0.4724
⏱️  ETA: 02h 36m remaining

📈 Step 400/7,782 (5.1%)
💔 Loss: 0.4245
⏱️  ETA: 02h 34m remaining

📈 Step 500/7,782 (6.4%)
💔 Loss: 0.2497
⏱️  ETA: 02h 32m remaining

📈 Step 600/7,782 (7.7%)
💔 Loss: 0.4419
⏱️  ETA: 02h 30m remaining

📈 Step 700/7,782 (9.0%)
💔 Loss: 0.3136
⏱️  ETA: 02h 28m remaining

📈 Step 800/7,782 (10.3%)
💔 Loss: 0.2572
⏱️  ETA: 02h 26m

In [ ]:
#cell 30:
# 🧩 Step 10 — Save Model

print("💾 Saving model...")

import os

# Create output directory
output_dir = "./spoc_gpt2_finetuned"
os.makedirs(output_dir, exist_ok=True)

# Save the fine-tuned model
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved successfully to: {output_dir}")

# List saved files
print("\n📁 Saved files:")
for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
    print(f"  - {file} ({size:.2f} MB)")

print("\n🎉 Model saving completed!")